In [7]:
import numpy as np
from scipy.stats import ranksums


def compute_statistics(data):
    arr = np.array(data)
    return {
        "mean": np.mean(arr),
        "var": np.var(arr, ddof=1),
        "std": np.std(arr, ddof=1),
        "median": np.median(arr),
        "min": np.min(arr),
        "max": np.max(arr),
    }


def compare_algorithms(rmse_a, rmse_b, alpha=0.05):
    """
    Сравнивает два набора значений RMSE (списки чисел) с помощью
    двустороннего критерия Вилкоксона (ранговой суммы).

    Параметры:
        rmse_a : list or array
            Значения RMSE для первого алгоритма (например, DECC).
        rmse_b : list or array
            Значения RMSE для второго алгоритма (например, ADAM).
        alpha : float, optional
            Уровень значимости для теста (по умолчанию 0.05).

    Возвращает:
        dict с полями:
            stats_a    : dict – статистики для первого набора
            stats_b    : dict – статистики для второго набора
            p_value    : float – p‑значение критерия Вилкоксона
            significant: bool – True, если p_value < alpha
    """
    # Проверка на пустые входные данные
    if len(rmse_a) == 0 or len(rmse_b) == 0:
        raise ValueError("Оба списка должны содержать хотя бы одно значение")

    # Статистики по каждому алгоритму
    stats_a = compute_statistics(rmse_a)
    stats_b = compute_statistics(rmse_b)

    # Критерий Вилкоксона (ранговой суммы)
    # ranksums возвращает кортеж (statistic, p-value)
    _, p_value = ranksums(rmse_a, rmse_b)

    significant = p_value < alpha

    return {
        "stats_a": stats_a,
        "stats_b": stats_b,
        "p_value": p_value,
        "significant": significant,
    }

In [ ]:
import pandas as pd
import sklearn.metrics as metrics

file_names = {
    "I_6_2b.txt",
    "I_8_14.txt",
    "I_12_1.txt",
    "I_12_2.txt",
    "I_12_4.txt",
    "I_14_3.txt",
    "I_14_4.txt",
    "I_15_3x.txt",
    "I_15_10.txt",
    "I_18_4.txt",
    "I_24_6.txt",
    "I_34_8.txt",
}
parametrs_count = {3, 4, 2, 4, 3, 3, 2, 4, 3, 4, 4, 4}
layerCount = {2, 3, 4}
neuronCount = {4, 3, 2}


data = pd.read_csv(
    "results_DECC/I_6_2b_test_fevG8_T5_lCount3_nCount3_run1.txt", sep="\t"
)

print(metrics.root_mean_squared_error(data["Y_true"], data["Y_pred"]))
# compute_statistics(data[])

0.034556590127846815


In [4]:
# Пример данных (реальные значения из вашего файла)
rmse_decc = [
    0.165348,
    0.258873,
    0.177173,
    0.257798,
    0.160783,
    0.160818,
    0.160156,
    0.160751,
    0.160844,
    0.160101,
    0.175479,
    0.243683,
    0.257752,
    0.181374,
    0.160549,
    0.162421,
    0.252543,
    0.157003,
    0.244277,
]
rmse_adam = [
    0.160,
    0.250,
    0.170,
    0.260,
    0.155,
    0.158,
    0.159,
    0.162,
    0.163,
    0.159,
    0.172,
    0.240,
    0.255,
    0.180,
    0.161,
    0.163,
    0.250,
    0.156,
    0.245,
]  # приблизительные значения

result = compare_algorithms(rmse_decc, rmse_adam, alpha=0.05)
print("Статистики DECC:", result["stats_a"])
print("Статистики ADAM:", result["stats_b"])
print("p-значение:", result["p_value"])
print("Статистически значимое различие:", result["significant"])

Статистики DECC: {'mean': np.float64(0.19251189473684213), 'var': np.float64(0.0018066495502105265), 'std': np.float64(0.04250470033079314), 'median': np.float64(0.165348), 'min': np.float64(0.157003), 'max': np.float64(0.258873)}
Статистики ADAM: {'mean': np.float64(0.19042105263157894), 'var': np.float64(0.001776701754385965), 'std': np.float64(0.04215094013644257), 'median': np.float64(0.163), 'min': np.float64(0.155), 'max': np.float64(0.26)}
p-значение: 0.609415426248741
Статистически значимое различие: False


In [1]:
# Ячейка 1: Импорт и настройка
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
from io import StringIO

# Укажите путь к данным
DATA_PATH = "build/results/"  # Измените на ваш путь


# Исходная функция
def original_function(x):
    return 2.5 * np.sin(1.1 * np.cos(1.1 * x + 2) * x + 5) + 7.3


# Ячейка 2: Функция загрузки данных
def load_data_with_comma_support(file_path):
    """Загружает данные, поддерживая оба формата десятичных разделителей"""
    try:
        data = np.loadtxt(file_path)
        return data
    except ValueError as e:
        if "could not convert string" in str(e):
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()
                content = content.replace(",", ".")
                data = np.loadtxt(StringIO(content))
                return data
            except:
                return None
        return None
    except:
        return None


# Ячейка 3: Нахождение всех медианных прогонов
# Параметры из вашего кода
fevGlobal_list = [10, 8, 5]
T_list = [4, 5, 8]
layerCount_list = [2, 3, 4]
neuronCount_list = [4, 3, 2]

median_runs_info = []

# Проходим по всем комбинациям параметров
for ft in range(3):
    fevGlobal = fevGlobal_list[ft]
    T = T_list[ft]

    for ln in range(3):
        layerCount = layerCount_list[ln]
        neuronCount = neuronCount_list[ln]

        rmses = []
        run_numbers = []

        # Собираем RMSE для всех прогонов (1-7)
        for runNumber in range(1, 8):
            test_filename = f"results_test_fevG{fevGlobal}_T{T}_lCount{layerCount}_nCount{neuronCount}_run{runNumber}.txt"
            test_path = Path(DATA_PATH) / test_filename

            if test_path.exists():
                test_data = load_data_with_comma_support(test_path)
                if test_data is not None and len(test_data) > 0:
                    if test_data.ndim == 2 and test_data.shape[1] >= 3:
                        testY = test_data[:, 1]
                        pred = test_data[:, 2]
                        rmse = np.sqrt(np.mean((testY - pred) ** 2))
                        rmses.append(rmse)
                        run_numbers.append(runNumber)

        # Находим медианный прогон
        if rmses:
            sorted_indices = np.argsort(rmses)
            sorted_rmses = np.array(rmses)[sorted_indices]
            sorted_runs = np.array(run_numbers)[sorted_indices]

            median_idx = len(sorted_rmses) // 2
            median_run = sorted_runs[median_idx]
            median_rmse = sorted_rmses[median_idx]

            median_runs_info.append(
                {
                    "fevGlobal": fevGlobal,
                    "T": T,
                    "layerCount": layerCount,
                    "neuronCount": neuronCount,
                    "median_run": median_run,
                    "median_rmse": median_rmse,
                    "total_runs": len(rmses),
                }
            )

print(f"✅ Найдено {len(median_runs_info)} комбинаций с медианными прогонами")


# Ячейка 4: Функция построения графика для одной комбинации
def plot_single_combo(combo_info):
    """Строит график для одной комбинации параметров"""

    fevGlobal = combo_info["fevGlobal"]
    T = combo_info["T"]
    layerCount = combo_info["layerCount"]
    neuronCount = combo_info["neuronCount"]
    median_run = combo_info["median_run"]

    # Загружаем данные
    test_filename = f"results_test_fevG{fevGlobal}_T{T}_lCount{layerCount}_nCount{neuronCount}_run{median_run}.txt"
    train_filename = f"results_train_fevG{fevGlobal}_T{T}_lCount{layerCount}_nCount{neuronCount}_run{median_run}.txt"

    test_data = load_data_with_comma_support(Path(DATA_PATH) / test_filename)
    train_data = load_data_with_comma_support(Path(DATA_PATH) / train_filename)

    if test_data is None or train_data is None:
        print(f"❌ Не удалось загрузить данные для этой комбинации")
        return

    # Извлекаем данные
    if test_data.ndim == 1:
        testX, testY, test_pred = test_data[0], test_data[1], test_data[2]
    else:
        testX, testY, test_pred = test_data[:, 0], test_data[:, 1], test_data[:, 2]

    if train_data.ndim == 1:
        trainX, trainY, train_pred = train_data[0], train_data[1], train_data[2]
    else:
        trainX, trainY, train_pred = (
            train_data[:, 0],
            train_data[:, 1],
            train_data[:, 2],
        )

    # Определяем диапазон для построения
    all_x = np.concatenate([trainX.flatten(), testX.flatten()])
    x_min, x_max = all_x.min(), all_x.max()
    x_range = x_max - x_min
    x_min_plot = x_min - 0.1 * x_range
    x_max_plot = x_max + 0.1 * x_range

    # Создаем точки для исходной функции
    x_plot = np.linspace(x_min_plot, x_max_plot, 1000)
    y_plot = original_function(x_plot)

    # Сортируем для построения линии предсказаний
    sorted_idx = np.argsort(testX)
    testX_sorted = testX[sorted_idx]
    test_pred_sorted = test_pred[sorted_idx]

    # Создаем график
    fig, ax = plt.subplots(figsize=(12, 8))

    # Исходная функция
    ax.plot(x_plot, y_plot, "k-", linewidth=3, alpha=0.7, label="Исходная функция")

    # Обучающая выборка
    ax.scatter(
        trainX,
        trainY,
        color="blue",
        s=80,
        alpha=0.6,
        label=f"Обучающая выборка (n={len(trainX)})",
    )

    # Тестовая выборка
    ax.scatter(
        testX,
        testY,
        color="green",
        s=100,
        alpha=0.6,
        marker="s",
        label=f"Тестовая выборка (n={len(testX)})",
    )

    # Построенная зависимость
    ax.plot(
        testX_sorted,
        test_pred_sorted,
        "r-",
        linewidth=2,
        label="Построенная зависимость",
    )
    ax.scatter(
        testX,
        test_pred,
        color="red",
        s=60,
        alpha=0.6,
        marker="^",
        label="Предсказанные значения",
    )

    # Настройки графика
    ax.set_title(
        f"fevGlobal={fevGlobal}, T={T}, Layers={layerCount}, Neurons={neuronCount}\n"
        f'Медианный прогон: {median_run}, RMSE={combo_info["median_rmse"]:.4f}',
        fontsize=14,
        pad=15,
    )
    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("y", fontsize=12)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.legend(loc="best", fontsize=10)

    # Добавляем подпись с информацией
    plt.figtext(
        0.02,
        0.02,
        f'Всего прогонов: {combo_info["total_runs"]}/7',
        fontsize=9,
        style="italic",
        alpha=0.7,
    )

    plt.tight_layout()
    plt.show()


# Ячейка 5: Построение всех графиков по отдельности
print("📊 Построение графиков для каждой комбинации:")
print("=" * 50)

for i, combo in enumerate(median_runs_info, 1):
    print(
        f"\nГрафик {i}: fevG={combo['fevGlobal']}, T={combo['T']}, "
        f"L={combo['layerCount']}, N={combo['neuronCount']}"
    )
    plot_single_combo(combo)

# Сохраняем информацию
df_median = pd.DataFrame(median_runs_info)
df_median.to_csv("median_runs_jupyter.csv", index=False)
print(f"\n💾 Информация сохранена в 'median_runs_jupyter.csv'")

✅ Найдено 0 комбинаций с медианными прогонами
📊 Построение графиков для каждой комбинации:

💾 Информация сохранена в 'median_runs_jupyter.csv'
